# データとtoken数の目視確認

最初のconfiguration cellを変更し、**Restart Kernel and Run All Cells** で上から実行してください。
既存のraw reader・inventory・preprocessing・token counterを呼ぶ閲覧用Notebookです。
data/の取得物、既存CSV、batch結果には書き込みません。OpenAI API・LLM評価・simulationは実行しません。

一覧は公式20 task＋外部5候補。Baiduは未取得、公式9 taskはrawのみで元promptは未対応です。
Notebookの保存出力は実行時の設定に対応するsnapshotなので、設定変更後は全セルを再実行してください。


In [ ]:
DATASET = "MountainCar-v0"
EPISODE = 0
QUERY_INDEX = 10
HISTORY_SIZE = 1
HISTORY_SIZES = [0, 1, 2, 3, 5]

# None / None: 選択taskの既存LLM-X既定質問を使用。
# 指定する場合は両方変更。例: "next-action", "next_action_prediction"
METRIC = None
QUESTION_NAME = None
MODEL = "gpt-4o"

# 外部trajectory: SEQUENCE_IDとQUERY_INDEX。static: ROW_ID（全reader単位の通し番号）。
SEQUENCE_ID = None
ROW_ID = None

# 外部token表示は明示的に指定した場合だけ。Noneならraw inspectionのみ。
# 例: "configs/preprocessing/f16capstone_default.yaml"
PREPROCESSING_CONFIG = None
EXTERNAL_FULL_DATASET = False  # Trueで明示設定scope全体。既定は選択sequenceのみ。
MAX_EXTERNAL_QUERIES = 100_000 # 上限を超える計測は実行しない（sampleを全件に見せない）。


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# workspace直下／notebooks/のどちらから起動しても同じ既存moduleをimport。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dataset_adapters").is_dir() and (p / "configs/sources.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("workspace直下またはnotebooks/から起動してください。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataset_adapters.registry import get_adapter, EXTERNAL
from tools.notebook_inspection import (
    dataset_table, select_record, inspect_selected_file, preview_raw,
    preprocessing_for_inspection, selected_query, token_record,
    dataset_token_table, table_statistics, read_batch_summary,
)
from tools.count_input_tokens import resolve_encoding

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
print("Workspace:", ROOT)
print("Read-only inspection. No API client or API request.")


## 1. 全dataset一覧

既存inventoryを使うlist_datasets.pyの共通処理から作成します。外部の固定state/action/reward shapeは未定義（null）で、欠損値を0にしません。外部record数は異種stream・duplicate export等を含むreader件数です。


In [ ]:
datasets = dataset_table()
overview_columns = [
    "dataset_id", "category", "status", "files", "sequences", "records",
    "state_shape", "state_dtype", "action_shape", "action_dtype",
    "reward_shape", "reward_dtype", "sequential", "prompt_mode",
]
with pd.option_context("display.max_rows", None):
    display(datasets[overview_columns])
print(f"{len(datasets)} datasets; Baidu: NOT_ACQUIRED")


## 2. 選択datasetのschema — RAW DATA

選択元ファイルを既存inventoryコードで再読込し、保存済みinventoryのshape/dtypeと並べます。
shapeのsingleton次元を省略しません。CSVにはネイティブ数値dtypeがないため、schemaはinventory readerの型解釈、raw valueはreaderが保持した元の文字列として区別します。

TrajAirの配布元processed TXTは取得物の一部で、本projectがrawへ加工保存したものではありません。
F16のopaque MATLAB object・モデル等はraw adapter非対応のままです。


In [ ]:
adapter = get_adapter(DATASET)
description = adapter.describe()
display(pd.DataFrame([
    {"item": "Dataset", "value": DATASET},
    {"item": "Source", "value": description["source"]},
    {"item": "Format", "value": description["format"]},
    {"item": "Files", "value": description["number_of_files"]},
    {"item": "Episodes / trajectories", "value": description["number_of_sequences"]},
    {"item": "Timesteps / rows (reader scope)", "value": description["number_of_records"]},
]))
sequence, raw_record = select_record(
    adapter, episode=EPISODE, sequence_id=SEQUENCE_ID, row_id=ROW_ID, index=QUERY_INDEX
)
schema_table = pd.DataFrame()
if sequence is None:
    print("NOT_ACQUIRED: raw data inspection unavailable.")
else:
    print("Source file:", sequence.source_file)
    print("Sequence:", sequence.sequence_id, "key:", sequence.key)
    print("Selected reader-unit records:", sequence.length)
    schema_table, file_counts, actual_schema = inspect_selected_file(adapter, sequence)
    print("Actual file read vs saved inventory (file-level schema):")
    with pd.option_context("display.max_rows", None):
        display(schema_table)
    display(file_counts)
    if description.get("unreadable_variables"):
        print("Opaque MATLAB variables are inventory-only; not converted to trajectories.")
        display(pd.DataFrame(description["unreadable_variables"])[["source_file", "variable", "reason"]])


## 3. 実際のraw value — RAW DATA

先頭5 recordと指定番号を表示します。公式はstates[:5] / actions[:5] / rewards[:5]を含む全arrayを表示。
episodic_returnはepisode全体の値として別扱いです。外部では元columnをそのまま表示し、state/action/rewardへ変換しません。

外部の別ファイルを選ぶには `adapter.sequences()` のsequence_idをconfigurationへ設定できます。


In [ ]:
raw_preview = None
if sequence is not None:
    raw_preview = preview_raw(adapter, sequence, count=5)
    if isinstance(raw_preview, dict):
        for name, values in raw_preview.items():
            print(f"\n{name} (episode scope)" if name == "episodic_return" else f"\n{name}[:5]")
            print(values)
    else:
        display(raw_preview)
    print("\nSELECTED RAW RECORD")
    print("episode =", raw_record["episode_id"], "sequence =", raw_record["sequence_id"],
          "row =", raw_record.get("row_id"), "index =", raw_record["index"])
    print("Source:", raw_record["source_file"])
    print("SHA256:", raw_record["source_file_sha256"])
    # JSON表示なので指定recordの値をDataFrameの省略表示に隠しません。
    print(json.dumps(raw_record["fields"], ensure_ascii=False, indent=2))
else:
    print("Raw value: N/A (not acquired)")


## 4. MODEL INPUT AFTER PREPROCESSING

ここからraw表示とは別です。公式対応taskはpreprocessing/llmx_original.pyを通し、元の
Episode / history_text / render_question / system_prompt / user_promptをそのまま使います。
未対応taskのpromptは作りません。

外部はPREPROCESSING_CONFIGを明示した場合だけ、その既存baselineを使います。
このpreprocessingはraw datasetそのものではなく、本projectで定義したLLM入力用変換です。
staticはH=0のみ。存在しないaction/rewardを生成しません。


In [ ]:
config, preprocessing_status = preprocessing_for_inspection(DATASET, PREPROCESSING_CONFIG)
print("Raw data inspection:", "available" if sequence is not None else "unavailable")
print(preprocessing_status)
query = None
if config is not None and sequence is not None:
    display(config)
    try:
        query = selected_query(adapter, sequence, raw_record["index"], config,
                               HISTORY_SIZE, METRIC, QUESTION_NAME)
    except (ValueError, IndexError) as exc:
        print("Selected model input: N/A —", exc)
    if query is not None:
        print("Prompt mode:", config["prompt_mode"])
        print("Metric / question:", query.metric, "/", query.question_name)
        print("Effective history size:", query.history_size)
        print("History interval [start, end):", query.history_start, query.history_end)
        print("\nHISTORY REPRESENTATION\n")
        print(query.history_text)
else:
    print("Token count: N/A")


## 5. 実際のprompt全文

以下が実際のSYSTEM PROMPTとUSER PROMPTです。外部設定時は元LLM-Xではなくexternal baselineと表示します。文字列を組み立てるだけで送信はしません。


In [ ]:
if query is not None:
    print("Original LLM-X" if config["prompt_mode"] == "original_llmx" else "EXTERNAL BASELINE")
    print("\nSYSTEM PROMPT\n")
    print(query.system_prompt)
    print("\nUSER PROMPT\n")
    print(query.user_prompt)
else:
    print("Prompt: N/A")


## 6. 選択queryのtoken数

既存count_input_tokens.count_queryを使用。contentは実文字列のlocal tokenize結果です。
estimated_api_input_tokensは既存CLIと同じcontent + 2×3 + 3の概算で、server usageではありません。
history/questionを別々にtokenizeした値はBPE境界のためuser_tokensへ単純加算できません。
未知のmodel名ではfallbackを明示します。tokenizer未キャッシュ時だけ公開tokenizer assetの取得が必要になる場合があります（inference APIではありません）。


In [ ]:
selected_tokens = None
if query is not None:
    encoding, fallback_used = resolve_encoding(MODEL)
    print("Model:", MODEL, "\nTokenizer:", encoding.name)
    print("Fallback tokenizer:", fallback_used)
    if fallback_used:
        print("WARNING: unknown model; fallback tokenizer is being used.")
    selected_tokens = token_record(query, sequence, config, encoding, fallback_used, MODEL)
    token_columns = ["system_tokens", "history_tokens", "question_tokens", "user_tokens",
                     "total_content_tokens", "estimated_api_input_tokens"]
    display(pd.DataFrame([{"component": key, "tokens": selected_tokens[key]} for key in token_columns]))
    print("Preprocessing SHA256:", selected_tokens["preprocessing_config_sha256"])
else:
    print("Token count: N/A")


## 7. データ番号ごとのtoken数

公式は選択datasetの**全episode・全valid query**を、現在のH / metric / questionで再計測します。
全行はquery_tokens DataFrameに保持し、長い表の画面表示は先頭・末尾に省略されます。
外部は既定で選択sequenceのみ。全設定scopeはEXTERNAL_FULL_DATASET=Trueで明示し、MAX_EXTERNAL_QUERIESを超えた場合は実行しません。自動samplingはしません。


In [ ]:
query_tokens, count_scope = dataset_token_table(
    adapter, config, HISTORY_SIZE, MODEL, METRIC, QUESTION_NAME,
    external_sequence=sequence.sequence_id if sequence else None,
    external_full_dataset=EXTERNAL_FULL_DATASET,
    max_external_queries=MAX_EXTERNAL_QUERIES,
)
print("Scope:", count_scope)
print("Computed query rows:", len(query_tokens))
query_columns = ["episode_id", "sequence_id", "query_index", "history_start", "history_end",
                 "input_tokens", "source_file"]
if not query_tokens.empty:
    display(query_tokens[query_columns])
    if query_tokens["fallback_encoding_used"].any():
        print("WARNING: fallback tokenizer used:", query_tokens["tokenizer"].unique())


## 8. token統計・上位5件／下位5件

既存token_statisticsを利用。stdは母標準偏差(ddof=0)、p95はnearest-rankです。
input_tokensは上記のAPI入力概算を指します。


In [ ]:
statistics_table = pd.DataFrame()
if not query_tokens.empty:
    statistics_table = pd.DataFrame([table_statistics(query_tokens)])
    display(statistics_table)
    print("Largest 5 queries")
    display(query_tokens.nlargest(5, "input_tokens")[query_columns])
    print("Smallest 5 queries")
    display(query_tokens.nsmallest(5, "input_tokens")[query_columns])
else:
    print("Statistics: N/A")


## 9. token分布

In [ ]:
if not query_tokens.empty:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(query_tokens["input_tokens"], bins=30)
    ax.set_xlabel("Estimated API input tokens")
    ax.set_ylabel("Query count")
    ax.set_title(f"{DATASET} | H={query_tokens['history_size'].iloc[0]}")
    fig.tight_layout()
    plt.show()
else:
    print("Histogram: N/A")


## 10. history size比較

選択datasetに対してのみ計算します。各Hで有効query集合が変わるため、totalの差と同一queryの増加は別です。
各Hの全行はhistory_tables[H]に保持します。比較にはepisode_id / sequence_id / query_indexでjoinできます。
外部staticはH=0のみ表示します。


In [ ]:
history_tables = {}
history_rows = []
if config is not None:
    hs = [0] if config["prompt_mode"] == "external_static" else sorted(set(HISTORY_SIZES))
    for h in hs:
        current_h = 0 if config["prompt_mode"] == "external_static" else HISTORY_SIZE
        if h == current_h:
            table, scope = query_tokens, count_scope
        else:
            table, scope = dataset_token_table(
                adapter, config, h, MODEL, METRIC, QUESTION_NAME,
                external_sequence=sequence.sequence_id if sequence else None,
                external_full_dataset=EXTERNAL_FULL_DATASET,
                max_external_queries=MAX_EXTERNAL_QUERIES,
            )
        history_tables[h] = table
        if table.empty:
            print(f"H={h}: no queries — {scope}")
            continue
        history_rows.append({"history_size": h, **table_statistics(table), "scope": scope})
history_comparison = pd.DataFrame(history_rows)
display(history_comparison if not history_comparison.empty else "History comparison: N/A")


## 11. 保存済み全dataset token summary

outputs/token_counts/token_inventory.csvがあれば読み込むだけです。model/tokenizerの保存時情報も表示します。
**この表は現在のconfigurationで再計算した結果ではありません。**
prompt_mode・H・config SHA・全件/sample scopeを維持し、混ぜて合計しません。
CSVがなければbatchを起動せずメッセージだけ表示します。


In [ ]:
batch_summary = read_batch_summary()
if batch_summary is None:
    print("Token batch has not been executed yet.")
else:
    metadata_path = ROOT / "outputs/token_counts/summary.json"
    if metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text())
        print("Saved batch model:", metadata.get("model"), "tokenizer:", metadata.get("tokenizer"))
    batch_columns = [
        "dataset", "n_units", "prompt_mode", "history_size", "count_scope",
        "mean_input_tokens", "std_input_tokens", "min_input_tokens", "max_input_tokens",
        "p95_input_tokens", "total_input_tokens", "preprocessing_config_sha256",
    ]
    with pd.option_context("display.max_rows", None):
        display(batch_summary[[c for c in batch_columns if c in batch_summary]])
    print("No batch was started. Existing results were read only.")


### 処理の所在

- 一覧: tools/list_datasets.py、保存済みoutputs/dataset_inventory/
- raw読込: dataset_adapters/、再照合: tools/inventory_llmx_data.py / inventory_candidates.py
- prompt: preprocessing/llmx_original.py / prompting.py（Notebook内では再実装しない）
- local token / 統計: tools/count_input_tokens.py
- 閲覧用の表への整形: tools/notebook_inspection.py

参考: [OpenAI GPT-4o](https://developers.openai.com/api/docs/models/gpt-4o)、[Text generation](https://developers.openai.com/api/docs/guides/text)。
ここではモデル推論もAPI呼び出しも行っていません。
